In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
import plotly.express as px

In [10]:
# Current working directory (should be project_root/notebooks)
NOTEBOOK_DIR = Path.cwd()

# Go up one level to project root
PROJECT_ROOT = NOTEBOOK_DIR.parent

# Build path to CSV
FILE_PATH = PROJECT_ROOT / "data" / "viz_data" / "log_preds_merge.csv"

print("Loading from:", FILE_PATH)

df = pd.read_csv(FILE_PATH)

print("Loaded shape:", df.shape)

Loading from: /Users/ogdata-mpb16/Documents/acled_ukraine/data/viz_data/log_preds_merge.csv
Loaded shape: (15378, 13)


In [ ]:
# =========================
# 1) Validate required columns
# =========================
required_cols = [
    "actual_fatalities",
    "pred_fatalities",
    "latitude",
    "longitude",
]

missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}\nAvailable columns: {list(df.columns)}")

# Ensure numeric types
for col in ["actual_fatalities", "pred_fatalities", "latitude", "longitude"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Drop rows that can't be mapped
df = df.dropna(subset=["latitude", "longitude", "actual_fatalities", "pred_fatalities"]).copy()

print("Rows after NA drop:", len(df))


Rows after NA drop: 15378


In [ ]:
# =========================
# 2) Remove structural zero noise
# =========================

# Keep events where either actual OR predicted > 0
df = df[
    (df["actual_fatalities"] > 0) |
    (df["pred_fatalities"] > 0)
].copy()

print("Rows after zero-zero filter:", len(df))

Rows after zero-zero filter: 3935


In [ ]:
# =========================
# 3) Create error features
# =========================

df["error"] = df["pred_fatalities"] - df["actual_fatalities"]
df["abs_error"] = df["error"].abs()

# Optional: percent error (stable version)
df["pct_error"] = df["error"] / (df["actual_fatalities"] + 1)

print("Error summary:")
print(df["error"].describe())

Error summary:
count    3935.000000
mean       -4.851334
std        12.656263
min      -217.000000
25%        -3.000000
50%        -1.000000
75%         0.000000
max         8.000000
Name: error, dtype: float64


In [31]:
# =========================
# 4) Feature engineering for mapping
# =========================
df["error"] = df["pred_fatalities"] - df["actual_fatalities"]
df["abs_error"] = df["error"].abs()

# Helpful for skewed casualty distributions (optional)
df["pct_error"] = df["error"] / (df["actual_fatalities"] + 1)

# Optional: log-space error (if you have log_pred_fatalities already)
if "log_pred_fatalities" in df.columns:
    df["log_pred_fatalities"] = pd.to_numeric(df["log_pred_fatalities"], errors="coerce")
    df["log_actual_fatalities"] = np.log1p(df["actual_fatalities"])
    df["log_error"] = df["log_pred_fatalities"] - df["log_actual_fatalities"]

In [32]:
# =========================
# 5) Plotly scatter map
# =========================
hover_cols = []
for c in ["actual_fatalities", "pred_fatalities", "error", "abs_error", "sub_event_type", "admin1", "interaction"]:
    if c in df.columns:
        hover_cols.append(c)

# Ukraine center (roughly)
center_lat, center_lon = 49.0, 31.0

fig = px.scatter_mapbox(
    df,
    lat="latitude",
    lon="longitude",
    color="error",
    size="abs_error",
    size_max=18,
    hover_data=hover_cols,
    color_continuous_scale="RdBu",
    range_color=(-20, 10),   # 👈 clamp scale
    mapbox_style="carto-positron",
    zoom=5,
    center={"lat": 49, "lon": 31},
)

fig.update_layout(
    title="Predicted vs Actual Fatalities (Residual Map)",
    coloraxis_colorbar=dict(title="Error (Pred - Actual)"),
    margin=dict(l=0, r=0, t=40, b=0),
)

fig.show();

In [33]:
# By sub_event_type
# The top three sub_event_type values are represented, all others are represented as "Other"

major_types = [
    "Armed clash",
    "Air/drone strike",
    "Shelling/artillery/missile attack"
]

df["event_group"] = df["sub_event_type"].apply(
    lambda x: x if x in major_types else "Other"
)

hover_cols = ["actual_fatalities", "pred_fatalities", "error", "abs_error", "admin1", "sub_event_type"]

for event in sorted(df["event_group"].unique()):
    subset = df[df["event_group"] == event].copy()

    fig = px.scatter_mapbox(
        subset,
        lat="latitude",
        lon="longitude",
        color="error",
        size="abs_error",
        size_max=18,
        hover_data=hover_cols,
        color_continuous_scale="balance",
        range_color=(-20, 20),
        mapbox_style="carto-positron",
        zoom=5,
        center={"lat": 49, "lon": 31},
        title=f"Residual Map: {event}"
    )

    fig.show();